In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsssmartdata2698")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/olist_products_dataset.csv"

In [0]:
df_products = spark.read.option("header", True)\
                        .option("inferSchema", True)\
                        .csv(ruta)


In [0]:
products_schema = StructType(fields=[
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", DoubleType(), True),
    StructField("product_length_cm", DoubleType(), True),
    StructField("product_height_cm", DoubleType(), True),
    StructField("product_width_cm", DoubleType(), True)
])

In [0]:
df_products_final = spark.read\
.option("header", True)\
.schema(products_schema)\
.csv(ruta)


In [0]:
products_selected_df = df_products_final.select(
    col("product_id"),
    col("product_category_name"),
    col("product_name_lenght"),
    col("product_description_lenght"),
    col("product_photos_qty"),
    col("product_weight_g"),
    col("product_length_cm"),
    col("product_height_cm"),
    col("product_width_cm")
)

In [0]:
products_renamed_df = products_selected_df

In [0]:
products_final_df = products_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
products_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.products_raw")